# Vigilante — Personalized Email Importance Classifier Stage 0 - Baseline Classifier Demonstration

This notebook connects to a Gmail inbox (read-only, via OAuth), and classifies each
email as **important** or **unimportant** using a machine-learning model trained on a
real inbox.

**Model:** Logistic regression trained on 6,000 real emails, using engineered features
(sender type, unsubscribe headers, reply markers) combined with TF-IDF on subject lines.
Labels were derived via intent-based bucketing (transactional/personal = important,
marketing/newsletter = unimportant). Achieves ~0.90 recall on the important class.



In [2]:
# Install Google API libraries
!pip install --quiet google-auth google-auth-oauthlib google-api-python-client

import re, json, pickle
import numpy as np
import pandas as pd
import joblib
from scipy.sparse import hstack, csr_matrix
from google_auth_oauthlib.flow import Flow
from googleapiclient.discovery import build
from google.colab import files
print("Setup complete.")

Setup complete.


## Step 1 — Upload the trained model files

Upload the 4 model artifacts:
`vigilante_model2.pkl`, `vigilante_vectorizer2.pkl`, `vigilante_scaler2.pkl`, `vigilante_metadata2.json`

In [3]:
# Upload the 4 model files (select all 4 in the picker)
print("Upload: vigilante_model2.pkl, vigilante_vectorizer2.pkl, vigilante_scaler2.pkl, vigilante_metadata2.json")
files.upload()

# Load them
model      = joblib.load("vigilante_model2.pkl")
vectorizer = joblib.load("vigilante_vectorizer2.pkl")
scaler     = joblib.load("vigilante_scaler2.pkl")
with open("vigilante_metadata2.json") as f:
    META = json.load(f)

MARKETING_SENDER_HINTS = META["marketing_sender_hints"]
THRESHOLD = META["threshold"]
print("✓ Model loaded. Threshold:", THRESHOLD)

Upload: vigilante_model2.pkl, vigilante_vectorizer2.pkl, vigilante_scaler2.pkl, vigilante_metadata2.json


Saving vigilante_scaler2.pkl to vigilante_scaler2.pkl
Saving vigilante_vectorizer2.pkl to vigilante_vectorizer2.pkl
Saving vigilante_metadata2.json to vigilante_metadata2.json
Saving vigilante_model2.pkl to vigilante_model2.pkl
✓ Model loaded. Threshold: 0.5


## Step 2 — Connect to Gmail (read-only OAuth)

Upload your `credentials.json` (Google Cloud OAuth client), then authorize access.

In [4]:
# Upload credentials.json
print("Upload credentials.json")
files.upload()

# OAuth connect (read-only)
SCOPES = ["https://www.googleapis.com/auth/gmail.readonly"]
flow = Flow.from_client_secrets_file("credentials.json", scopes=SCOPES,
                                     redirect_uri="urn:ietf:wg:oauth:2.0:oob")
auth_url, _ = flow.authorization_url(prompt="consent")
print("\n1. Open this URL, approve access:\n")
print(auth_url)
code = input("\n2. Paste the authorization code here: ").strip()
flow.fetch_token(code=code)
service = build("gmail", "v1", credentials=flow.credentials)

profile = service.users().getProfile(userId="me").execute()
print("\n✓ Connected:", profile["emailAddress"])

Upload credentials.json


Saving credentials.json to credentials.json

1. Open this URL, approve access:

https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=326381206565-guvhep4c08vsnhr58g0vcq4as5e26ihf.apps.googleusercontent.com&redirect_uri=urn%3Aietf%3Awg%3Aoauth%3A2.0%3Aoob&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.readonly&state=BgxbIGOeMnkQvanCIXS8kwVwH3XUYP&code_challenge=oRAHXrVnZEQ6A5HHZsZAQ2LGgadkHepHn1D1Mt4UQyw&code_challenge_method=S256&prompt=consent&access_type=offline

2. Paste the authorization code here: 4/1ATsMZqAyQn8wc_iEW7oler9H89AztT53i525ZpVBP2PI8ZeKvB4LRIxgcmI

✓ Connected: aryanhere10@gmail.com


## Step 3 — Fetch inbox emails

In [5]:
def get_header(headers, name):
    for h in headers:
        if h["name"].lower() == name.lower():
            return h["value"]
    return ""

MAX_EMAILS = 50   # change as needed

refs = service.users().messages().list(
    userId="me", labelIds=["INBOX"], maxResults=MAX_EMAILS
).execute().get("messages", [])

emails = []
for ref in refs:
    msg = service.users().messages().get(
        userId="me", id=ref["id"], format="metadata",
        metadataHeaders=["From", "Subject", "List-Unsubscribe", "Precedence"],
    ).execute()
    h = msg["payload"].get("headers", [])
    emails.append({
        "sender": get_header(h, "From"),
        "subject": get_header(h, "Subject"),
        "unsubscribe": get_header(h, "List-Unsubscribe"),
        "precedence": get_header(h, "Precedence"),
        "snippet": msg.get("snippet", ""),
    })
print(f"✓ Fetched {len(emails)} emails.")

✓ Fetched 50 emails.


## Step 4 — Classify each email

In [6]:
def eng_features(e):
    sender = e["sender"]; subject = str(e["subject"]).lower()
    return [
        int(any(h in sender.lower() for h in MARKETING_SENDER_HINTS)),
        int(str(e["unsubscribe"]).strip() != ""),
        int(str(e["precedence"]).lower() in ("bulk", "list")),
        int(subject.startswith("re:") or subject.startswith("fwd:") or subject.startswith("fw:")),
        len(subject),
    ]

for e in emails:
    eng = scaler.transform(np.array([eng_features(e)], dtype=float))
    tf = vectorizer.transform([str(e["subject"])])
    x = hstack([csr_matrix(eng), tf]).tocsr()
    e["score"] = float(model.predict_proba(x)[0, 1])
    e["verdict"] = "IMPORTANT" if e["score"] >= THRESHOLD else "unimportant"

print("✓ Classified all emails.")

✓ Classified all emails.


## Final Results

In [8]:
# Step 5 — Results with summary

df = pd.DataFrame([{
    "verdict": e["verdict"],
    "score": round(e["score"], 3),
    "sender": e["sender"],
    "subject": e["subject"],
} for e in sorted(emails, key=lambda x: x["score"], reverse=True)])

# --- Summary statistics ---
total   = len(df)
n_imp   = (df["verdict"] == "IMPORTANT").sum()
n_unimp = total - n_imp
avg_imp   = df[df["verdict"] == "IMPORTANT"]["score"].mean() if n_imp else 0
avg_unimp = df[df["verdict"] == "unimportant"]["score"].mean() if n_unimp else 0

print("=" * 55)
print("           CLASSIFICATION SUMMARY")
print("=" * 55)
print(f"  Total emails classified : {total}")
print(f"  Important               : {n_imp:3d}  ({100*n_imp/total:.1f}%)")
print(f"  Unimportant             : {n_unimp:3d}  ({100*n_unimp/total:.1f}%)")
print("-" * 55)
print(f"  Avg score (important)   : {avg_imp:.3f}")
print(f"  Avg score (unimportant) : {avg_unimp:.3f}")
print("=" * 55)

# --- Confidence breakdown ---
high_conf_imp   = ((df["verdict"]=="IMPORTANT") & (df["score"]>=0.8)).sum()
high_conf_unimp = ((df["verdict"]=="unimportant") & (df["score"]<=0.2)).sum()
borderline      = ((df["score"]>0.4) & (df["score"]<0.6)).sum()
print(f"\n  High-confidence important  (score >= 0.8): {high_conf_imp}")
print(f"  High-confidence unimportant (score <= 0.2): {high_conf_unimp}")
print(f"  Borderline / uncertain     (0.4 - 0.6)   : {borderline}")

# --- Top important emails ---
print("\n" + "=" * 55)
print("  TOP 5 MOST IMPORTANT")
print("=" * 55)
for _, r in df.head(5).iterrows():
    print(f"  [{r['score']:.2f}] {r['sender'][:28]:<30} {r['subject'][:35]}")

# --- Save CSV ---
df.to_csv("classification_results.csv", index=False)
files.download("classification_results.csv")

# --- Full table ---
print("\nFull classified table below:")
pd.set_option("display.max_colwidth", 50)
df

           CLASSIFICATION SUMMARY
  Total emails classified : 50
  Important               :   7  (14.0%)
  Unimportant             :  43  (86.0%)
-------------------------------------------------------
  Avg score (important)   : 0.782
  Avg score (unimportant) : 0.109

  High-confidence important  (score >= 0.8): 3
  High-confidence unimportant (score <= 0.2): 37
  Borderline / uncertain     (0.4 - 0.6)   : 1

  TOP 5 MOST IMPORTANT
  [0.97] services@cdslindia.co.in       Transactions In Your Demat Account
  [0.97] services@cdslindia.co.in       Transactions In Your Demat Account
  [0.96] nse_alerts <nse_alerts@nse.c   Funds/Securities Balance
  [0.74] "McKinsey.org Forward" <forw   It's time to start your Forward jou
  [0.72] Unstop <noreply@emails.unsto   Here’s Your Unstop Verification Cod


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Full classified table below:


,verdict,score,sender,subject
0,IMPORTANT,0.966,services@cdslindia.co.in,Transactions In Your Demat Account
1,IMPORTANT,0.966,services@cdslindia.co.in,Transactions In Your Demat Account
2,IMPORTANT,0.965,nse_alerts <nse_alerts@nse.co.in>,Funds/Securities Balance
3,IMPORTANT,0.745,"""McKinsey.org Forward"" <forward@mail.mckinsey....",It's time to start your Forward journey. Apply...
4,IMPORTANT,0.724,Unstop <noreply@emails.unstop.com>,Here’s Your Unstop Verification Code.
5,IMPORTANT,0.603,"""McKinsey.org Forward"" <forward@mail.mckinsey....",Forward applications are now open. Apply today!
6,IMPORTANT,0.504,HDFC MUTUAL FUND <donotreplytoHDFCMF@camsonlin...,Important: Change in Base Expense Ratio (BER) ...
7,unimportant,0.338,Groww <noreply@groww.in>,Mutual fund: Redemption in progress
8,unimportant,0.327,Groww <noreply@groww.in>,Mutual fund: Redemption done
9,unimportant,0.286,Motilal Oswal Mutual Fund <noreply@motilaloswa...,Built Investor by Investor. Motilal Oswal Midc...
